# Cell2location Visium Deconvolution Tool Test

This notebook demonstrates how to exercise the `run_cell2location_visium_deconvolution` tool on the `V1_Human_Lymph_Node` Visium dataset alongside a matched single-cell reference.

> ⚠️ **Compute requirements**
>
> Training both the reference regression model and the spatial deconvolution model can take tens of minutes and benefits from GPU acceleration. Run the final cell only when you have provisioned adequate resources.

In [4]:
import urllib.request

import scanpy as sc
import sys

In [6]:
sys.path.append("/Users/wenduoc/TissueAgent/src")

In [7]:
from config import DATA_DIR, DATASET_DIR
from agents.agent_registry.single_cell_agent.tools_impl.cell2location_visium_deconvolution_tool import (
    run_cell2location_visium_deconvolution,
)

DATA_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Dataset directory: {DATASET_DIR}")

Data directory: /Users/wenduoc/TissueAgent/data
Dataset directory: /Users/wenduoc/TissueAgent/data/dataset


## 1. Download the Visium spatial dataset

The helper below downloads the `V1_Human_Lymph_Node` sample with spatial coordinates and image information. The result is stored inside the app's `data/dataset` directory.

In [8]:
sc.settings.datasetdir = str(DATA_DIR / "raw_datasets")
visium_adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")

visium_path = DATASET_DIR / "V1_Human_Lymph_Node_visium.h5ad"
visium_adata.write(visium_path)

print(f"Saved Visium AnnData to {visium_path}")

/var/folders/fs/3n9_hpnx7rv0vvh25klngrk80000gq/T/ipykernel_95483/3267265832.py:2: FutureWarning: Use `squidpy.datasets.visium` instead.
  visium_adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")


  0%|          | 0.00/7.86M [00:00<?, ?B/s]

  0%|          | 0.00/29.3M [00:00<?, ?B/s]

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scanpy/datasets/_datasets.py:555: FutureWarning: Use `squidpy.read.visium` instead.
  return read_visium(sample_dir, source_image_path=source_image_path)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/anndata/_core/anndata.py:1794: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/anndata/_core/anndata.py:1794: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Saved Visium AnnData to /Users/wenduoc/TissueAgent/data/dataset/V1_Human_Lymph_Node_visium.h5ad


## 2. Download the single-cell reference

Download the pre-trained single-cell reference AnnData object published alongside the cell2location manuscript.

In [9]:
reference_url = (
    "https://cell2location.cog.sanger.ac.uk/paper/integrated_lymphoid_organ_scrna/"
    "RegressionNBV4Torch_57covariates_73260cells_10237genes/sc.h5ad"
)
reference_path = DATASET_DIR / "cell2location_regression_reference.h5ad"

if not reference_path.exists():
    print(f"Downloading reference AnnData from {reference_url}")
    urllib.request.urlretrieve(reference_url, reference_path)
else:
    print(f"Reference AnnData already present at {reference_path}")

## 3. Run the cell2location Visium deconvolution tool

With the spatial and single-cell inputs in place, invoke the tool. The inputs are passed relative to `data/` so the helper resolves them automatically, and the output directory is created inside the same root.

In [11]:
visium_rel = visium_path.relative_to(DATA_DIR)
reference_rel = reference_path.relative_to(DATA_DIR)

result = run_cell2location_visium_deconvolution(
    visium_h5ad_path=str(visium_rel),
    reference_h5ad_path=str(reference_rel),
    output_subdir="cell2location_demo_run",
    cell_type_column="leiden",  # matches the clustering present in the published reference
    use_gpu=False,  # switch to False if GPU is unavailable
    regression_max_epochs=250,
    spatial_max_epochs=3000,
)
result

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/anndata/_core/anndata.py:1794: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


InvalidIndexError: Reindexing only valid with uniquely valued Index objects

## 4. Inspect generated outputs

When the run completes successfully, the dictionary above will contain the saved AnnData paths, trained model directories, and CSV exports of the estimated cell abundances. You can load the produced files with Scanpy or pandas for downstream analysis.